# Chapter 4: Stacks

A stack follows **Last In, First Out (LIFO)**: like a pile of plates, the last item added is the first removed. Items enter and leave at the **top**, while the oldest item sits at the **base**.

Stacks support undo history, browser navigation, function calls, expression parsing, backtracking, and depth-first search.

## Running this notebook

Select **JavaScript (Node.js)**, the same kernel used by the other chapter notebooks, then **Run All** from top to bottom. All implementations and examples live in this notebook; no local source files or extra npm packages are needed. Run the `Stack` definition before examples that use it. Example cells use local scopes so they can be rerun without declaration conflicts.

The supplied chapter uses TypeScript. These executable cells adapt its code to JavaScript: type annotations, interfaces, and imports are removed; the enum becomes a frozen object; and private fields use JavaScript's `#` syntax. The algorithms and stack APIs remain the same. In TypeScript, `Stack<T>` additionally restricts the types of values stored at compile time; these JavaScript examples do not provide that type checking.

## 1. LIFO in action

An array already provides `push()` and `pop()`. Watch the last book added leave first.

In [2]:
(() => {
  const books = [];
  books.push('JavaScript');
  books.push('TypeScript');
  books.push('Algorithms');
  console.log('Base → top:', books.join(' → '));
  while (books.length > 0) console.log('Remove:', books.pop());
})();

Base → top: JavaScript → TypeScript → Algorithms
Remove: Algorithms
Remove: TypeScript
Remove: JavaScript


## 2. Creating a reusable Stack class

A dedicated class exposes only stack operations and hides its backing array. `pop()` and `peek()` return `undefined` on an empty stack. `size` is a getter, so use `stack.size`, not `stack.size()`.

The class is defined here, without an export or an import. Later cells reuse it. Assigning the class to `var Stack` also allows this definition cell to be rerun.

In [4]:
var Stack = class Stack {
  #items = [];

  push(item) {
    this.#items.push(item);
  }

  pop() {
    return this.#items.pop();
  }

  peek() {
    return this.#items.at(-1);
  }

  isEmpty() {
    return this.#items.length === 0;
  }

  get size() {
    return this.#items.length;
  }

  clear() {
    this.#items = [];
  }

  toString() {
    if (this.isEmpty()) return 'Empty Stack';
    return this.#items
      .map((item) => typeof item === 'object' && item !== null
        ? JSON.stringify(item)
        : String(item))
      .join(', ');
  }
};

In [5]:
(() => {
  const stack = new Stack();
  console.log('Initially empty:', stack.isEmpty()); // true
  console.log('Empty pop:', stack.pop());           // undefined
  console.log('Empty peek:', stack.peek());         // undefined
  stack.push(10);
  stack.push(20);
  stack.push(30);
  console.log('Base → top:', stack.toString());     // 10, 20, 30
  console.log('Peek:', stack.peek());               // 30
  console.log('Size after peek:', stack.size);      // 3
  console.log('Pop:', stack.pop());                 // 30
  console.log('Size after pop:', stack.size);       // 2
  stack.clear();
  console.log('After clear:', stack.toString());    // Empty Stack
})();

Initially empty: true
Empty pop: undefined
Empty peek: undefined
Base → top: 10, 20, 30
Peek: 30
Size after peek: 3
Pop: 30
Size after pop: 2
After clear: Empty Stack


### Stack operations and cost

| Operation | Purpose | Typical time |
| --- | --- | --- |
| `push(item)` | Add to the top | O(1) amortized |
| `pop()` | Remove and return the top | O(1) |
| `peek()` | Read the top without removing it | O(1) |
| `isEmpty()` | Check whether there are any items | O(1) |
| `size` | Read the number of items | O(1) |
| `clear()` | Replace the internal array | O(1) assignment, excluding later garbage collection |
| `toString()` | Format the contents | O(n) for fixed-size items; object serialization depends on object size |

The stack uses O(n) storage. Its small API makes LIFO behavior explicit; arrays provide more general access.

## 3. Using a stack for undo history

The chapter's `EditorAction` interface describes an action's `type` and optional `text`. Here, plain JavaScript objects hold those fields. Every edit is pushed; undo pops the latest edit. This example records and removes actions; a complete editor would also reverse each action in its document.

In [ ]:
(() => {
  const ActionType = Object.freeze({
    Type: 'typing',
    Delete: 'deleting',
    Format: 'formatting',
  });
  const undoStack = new Stack();
  console.log(undoStack.isEmpty()); // true
  undoStack.push({ type: ActionType.Type, text: 'S' });
  undoStack.push({ type: ActionType.Type, text: 't' });
  undoStack.push({ type: ActionType.Type, text: 'a' });
  undoStack.push({ type: ActionType.Type, text: 'c' });
  undoStack.push({ type: ActionType.Type, text: 'k' });
  console.log(undoStack.peek()); // { type: 'typing', text: 'k' }
  console.log(undoStack.size);   // 5
  const lastAction = undoStack.pop();
  if (lastAction !== undefined) console.log('Undo:', lastAction);
  console.log(undoStack.size);   // 4
  console.log('Remaining actions:', undoStack.toString());
})();

## 4. Solving a maze with an array-based stack

Depth-first search (DFS) uses a stack to remember unexplored states. This example is independent of the `Stack` class: the array itself is the stack.

- `S` marks the start, `G` marks the goal, `0` is open, and `1` is a wall.
- A position is `[row, column]`; a state contains a position and the path taken to it.
- Moves go right, down, left, or up. The last eligible neighbor pushed is explored first.
- The goal test checks for `G`; a visited set prevents repeated exploration.

This is classical state-space search, not machine learning. DFS finds a reachable goal but **does not guarantee the shortest path**. Breadth-first search finds shortest paths in unweighted mazes; A* can use a heuristic.

Unlike the chapter's fixed `(0, 0)` start, this version locates `S`, so you can move it when experimenting. It returns `null` for an empty maze, a missing start, or an unreachable goal, and rejects uneven row lengths.

In [ ]:
var solveMaze = function solveMaze(grid) {
  const rows = grid.length;
  const columns = grid[0]?.length ?? 0;
  if (rows === 0 || columns === 0) return null;
  if (grid.some((row) => row.length !== columns)) {
    throw new Error('Maze rows must have equal lengths');
  }

  let start;
  for (let row = 0; row < rows; row++) {
    const column = grid[row].indexOf('S');
    if (column !== -1) {
      start = [row, column];
      break;
    }
  }
  if (start === undefined) return null;

  const stack = [];
  const visited = new Set();
  const directions = [[0, 1], [1, 0], [0, -1], [-1, 0]];
  stack.push({ row: start[0], column: start[1], path: [start] });

  while (stack.length > 0) {
    const current = stack.pop();
    if (current === undefined) break;
    const { row, column, path } = current;
    const key = `${row},${column}`;
    if (visited.has(key)) continue;
    visited.add(key);

    if (grid[row][column] === 'G') return path;

    for (const [rowChange, columnChange] of directions) {
      const nextRow = row + rowChange;
      const nextColumn = column + columnChange;
      const insideMaze = nextRow >= 0 && nextRow < rows
        && nextColumn >= 0 && nextColumn < columns;
      if (!insideMaze) continue;
      const nextKey = `${nextRow},${nextColumn}`;
      const isWall = grid[nextRow][nextColumn] === 1;
      if (!isWall && !visited.has(nextKey)) {
        stack.push({
          row: nextRow,
          column: nextColumn,
          path: [...path, [nextRow, nextColumn]],
        });
      }
    }
  }
  return null;
};

In [ ]:
(() => {
  const maze = [
    ['S', 0, 1, 0],
    [1, 0, 1, 0],
    [0, 0, 0, 0],
    [0, 1, 1, 'G'],
  ];
  const solution = solveMaze(maze);
  console.log('Path:', JSON.stringify(solution));
  // [[0,0],[0,1],[1,1],[2,1],[2,2],[2,3],[3,3]]
  console.log('Moves:', solution === null ? 'No route' : solution.length - 1);

  const route = new Set((solution ?? []).map(([row, column]) => `${row},${column}`));
  console.log('Route (*), walls (#), open cells (.):');
  console.log(maze.map((line, row) => line.map((cell, column) => {
    if (cell === 'S' || cell === 'G') return cell;
    if (cell === 1) return '#';
    return route.has(`${row},${column}`) ? '*' : '.';
  }).join(' ')).join('\n'));
  console.log('Unreachable goal:', solveMaze([['S', 1, 'G']])); // null
  console.log('Empty maze:', solveMaze([]));                  // null
})();

### Goals and search

The goal defines success, the available moves define possible actions, and DFS decides which state to explore next. The stack controls exploration order; it does not define the goal or make the program intelligent by itself.

This teaching implementation copies a path for each pushed state. Those copies add time and memory overhead beyond a basic DFS traversal. A larger solver can store parent links and reconstruct the path only after reaching the goal.

## 5. Converting a decimal number to binary

Repeated division by 2 produces remainders from least significant to most significant. Popping them from a stack reverses that order. Zero needs an explicit result.

Both conversion functions require **non-negative safe integers**. This tightens the chapter's integer check to avoid silently converting numbers beyond JavaScript's exact integer range.

In [ ]:
var decimalToBinary = function decimalToBinary(value) {
  if (!Number.isSafeInteger(value) || value < 0) {
    throw new Error('Value must be a non-negative safe integer');
  }
  if (value === 0) return '0';
  const remainders = new Stack();
  let dividend = value;
  while (dividend > 0) {
    remainders.push(dividend % 2);
    dividend = Math.floor(dividend / 2);
  }
  let result = '';
  while (!remainders.isEmpty()) {
    const remainder = remainders.pop();
    if (remainder !== undefined) result += remainder;
  }
  return result;
};

console.log(decimalToBinary(13)); // 1101
console.log(decimalToBinary(0));  // 0

## 6. Converting to bases 2 through 36

The same algorithm works with a different divisor. Digits above 9 use `A` through `Z`; base 16, for example, uses `0`–`9` and `A`–`F`.

In [ ]:
var decimalToBase = function decimalToBase(value, base) {
  if (!Number.isSafeInteger(value) || value < 0) {
    throw new Error('Value must be a non-negative safe integer');
  }
  if (!Number.isInteger(base) || base < 2 || base > 36) {
    throw new Error('Base must be an integer between 2 and 36');
  }
  if (value === 0) return '0';
  const digits = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ';
  const remainders = new Stack();
  let dividend = value;
  while (dividend > 0) {
    remainders.push(dividend % base);
    dividend = Math.floor(dividend / base);
  }
  let result = '';
  while (!remainders.isEmpty()) {
    const remainder = remainders.pop();
    if (remainder !== undefined) result += digits[remainder];
  }
  return result;
};

console.log(decimalToBase(100345, 2));  // 11000011111111001
console.log(decimalToBase(100345, 8));  // 303771
console.log(decimalToBase(100345, 16)); // 187F9
console.log(decimalToBase(100345, 35)); // 2BW0

In [ ]:
(() => {
  for (const [value, base] of [[-1, 2], [1.5, 2], [10, 1], [10, 37]]) {
    try {
      console.log(decimalToBase(value, base));
    } catch (error) {
      console.log(`value=${value}, base=${base}: ${error.message}`);
    }
  }
})();

## 7. Exercise: valid parentheses

Push each opening bracket. For each closing bracket, pop and check whether the pair matches. Success requires an empty stack at the end.

Try predicting the results before running the examples. This function accepts **bracket-only strings**: other characters are rejected, and the empty string is valid. Time and worst-case space are both O(n).

In [ ]:
var isValidParentheses = function isValidParentheses(input) {
  if (input.length % 2 !== 0) return false;
  const matchingBracket = { ')': '(', ']': '[', '}': '{' };
  const stack = [];
  for (const character of input) {
    if (character === '(' || character === '[' || character === '{') {
      stack.push(character);
      continue;
    }
    if (!Object.hasOwn(matchingBracket, character)) return false;
    if (stack.pop() !== matchingBracket[character]) return false;
  }
  return stack.length === 0;
};

console.log(isValidParentheses('()'));   // true
console.log(isValidParentheses('([])')); // true
console.log(isValidParentheses('(]'));   // false
console.log(isValidParentheses('([)]')); // false
console.log(isValidParentheses('(('));   // false
console.log(isValidParentheses(''));     // true
console.log(isValidParentheses('ab'));   // false

## 8. Exercise: minimum stack

A second stack tracks the minimum values seen so far. Push a new minimum when the value is **less than or equal to** the current minimum. Keeping equal minima is essential: popping one copy must leave the other available.

For finite numeric inputs, `top()` and `getMin()` are O(1), `pop()` is O(1), and array-backed `push()` is amortized O(1). Total storage is O(n). Empty reads return `undefined`.

In [ ]:
var MinStack = class MinStack {
  #values = [];
  #minimums = [];

  push(value) {
    this.#values.push(value);
    const currentMinimum = this.#minimums.at(-1);
    if (currentMinimum === undefined || value <= currentMinimum) {
      this.#minimums.push(value);
    }
  }

  pop() {
    const value = this.#values.pop();
    if (value !== undefined && value === this.#minimums.at(-1)) {
      this.#minimums.pop();
    }
    return value;
  }

  top() {
    return this.#values.at(-1);
  }

  getMin() {
    return this.#minimums.at(-1);
  }
};

In [ ]:
(() => {
  const minStack = new MinStack();
  minStack.push(-2);
  minStack.push(0);
  minStack.push(-3);
  console.log(minStack.getMin()); // -3
  minStack.pop();
  console.log(minStack.top());    // 0
  console.log(minStack.getMin()); // -2

  const duplicates = new MinStack();
  duplicates.push(2);
  duplicates.push(1);
  duplicates.push(1);
  duplicates.pop();
  console.log('One minimum remains:', duplicates.getMin()); // 1
  duplicates.pop();
  console.log('Previous minimum restored:', duplicates.getMin()); // 2
})();

## 9. Runnable checks

These assertions check LIFO order, empty behavior, maze path validity, conversions, bracket matching, and duplicate minima. A failed assertion throws an error in this cell.

In [ ]:
(() => {
  const assert = require('node:assert/strict');
  const stack = new Stack();
  assert.equal(stack.pop(), undefined);
  assert.equal(stack.peek(), undefined);
  [1, 2, 3].forEach((value) => stack.push(value));
  assert.equal(stack.peek(), 3);
  assert.equal(stack.size, 3);
  assert.deepEqual([stack.pop(), stack.pop(), stack.pop()], [3, 2, 1]);
  assert.equal(stack.isEmpty(), true);
  stack.push({ text: 'Stack' });
  assert.equal(stack.toString(), '{"text":"Stack"}');
  stack.clear();
  assert.equal(stack.size, 0);
  assert.equal(stack.toString(), 'Empty Stack');

  const grid = [['S', 0, 1, 0], [1, 0, 1, 0], [0, 0, 0, 0], [0, 1, 1, 'G']];
  const path = solveMaze(grid);
  assert.ok(path !== null);
  assert.deepEqual(path[0], [0, 0]);
  assert.deepEqual(path.at(-1), [3, 3]);
  assert.equal(new Set(path.map((position) => position.join(','))).size, path.length);
  path.forEach(([row, column], index) => {
    assert.ok(row >= 0 && row < grid.length && column >= 0 && column < grid[0].length);
    assert.notEqual(grid[row][column], 1);
    if (index > 0) {
      const [previousRow, previousColumn] = path[index - 1];
      assert.equal(Math.abs(row - previousRow) + Math.abs(column - previousColumn), 1);
    }
  });
  assert.equal(solveMaze([['S', 1, 'G']]), null);
  assert.equal(solveMaze([]), null);
  assert.equal(solveMaze([[]]), null);
  assert.equal(solveMaze([[0, 'G']]), null);
  assert.deepEqual(solveMaze([['G', 0, 'S']]), [[0, 2], [0, 1], [0, 0]]);
  assert.throws(() => solveMaze([['S', 0], ['G']]));

  for (const value of [0, 1, 13, 100345, Number.MAX_SAFE_INTEGER]) {
    assert.equal(decimalToBinary(value), value.toString(2));
    for (let base = 2; base <= 36; base++) {
      assert.equal(decimalToBase(value, base), value.toString(base).toUpperCase());
    }
  }
  for (const value of [-1, 1.5, NaN, Infinity, Number.MAX_SAFE_INTEGER + 1]) {
    assert.throws(() => decimalToBinary(value));
    assert.throws(() => decimalToBase(value, 2));
  }
  for (const base of [1, 37, 2.5, NaN]) assert.throws(() => decimalToBase(10, base));
  for (const input of ['', '()', '([])', '{[()]}', '()[]{}']) {
    assert.equal(isValidParentheses(input), true);
  }
  for (const input of ['(', '(]', '([)]', ')(', '((', 'ab']) {
    assert.equal(isValidParentheses(input), false);
  }

  const minimums = new MinStack();
  assert.equal(minimums.top(), undefined);
  assert.equal(minimums.getMin(), undefined);
  assert.equal(minimums.pop(), undefined);
  [2, 1, 1, 3].forEach((value) => minimums.push(value));
  for (const [value, nextMinimum] of [[3, 1], [1, 1], [1, 2], [2, undefined]]) {
    assert.equal(minimums.pop(), value);
    assert.equal(minimums.getMin(), nextMinimum);
  }
  console.log('All Chapter 4 checks passed.');
})();

## Summary and experiments

A stack is a LIFO abstraction. The reusable `Stack` class provides an encapsulated API, while arrays supply the same LIFO behavior directly through `push()` and `pop()`. Undo history and number conversion use the class; maze search and bracket validation use arrays; the minimum stack coordinates two arrays.

Try moving the maze's start or walls, changing the DFS direction order, converting a number to base 36, or adding bracket cases. Rerun the relevant cells to see how the results change.